# Run Baseline PFL Engine (Colab)

**Çalışma zamanı:** Üst menüden **Runtime → Change runtime type → GPU** (önerilir).
Bu notebook, tüm sweep mekanizmasını atlayarak doğrudan sadece saf `run_pfl.py` dosyasını yapılandırıp bir tek deneme (baseline) çalıştırır.

## 1) Projeyi getir

**Seçenek A — GitHub:** `REPO_URL` değerini kendi repona göre değiştir.

In [ ]:
import os
import subprocess

CONTENT = "/content"
REPO_URL = "https://github.com/mervegulec2/privproject.git"
BRANCH = "main"
PROJECT_ROOT = os.path.join(CONTENT, "privproject")

os.chdir(CONTENT)
if not os.path.isdir(PROJECT_ROOT):
    subprocess.check_call(["git", "clone", REPO_URL, "privproject"])
    os.chdir(PROJECT_ROOT)
    subprocess.check_call(["git", "checkout", BRANCH])
else:
    os.chdir(PROJECT_ROOT)
    subprocess.check_call(["git", "pull"])

os.environ["PROJECT_ROOT"] = os.getcwd()
print("PROJECT_ROOT:", os.environ["PROJECT_ROOT"])

**Seçenek B — Drive:** Zip yükleyip açıyorsanız (Clone yerine):

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
# import os
# os.environ["PROJECT_ROOT"] = "/content/drive/MyDrive/privproject"

## 2) Bağımlılıklar

Colab için `requirements.txt` kuruyoruz.

In [ ]:
import os
import subprocess
import sys

root = os.environ.get("PROJECT_ROOT", "/content/privproject")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=root,
)
# dotenv eksikse ayrıca kuralım ki .env dosyamızı kesinlikle okusun
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "python-dotenv"],
    cwd=root,
)
print("pip OK:", root)

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "flwr[simulation]",
    "ray[default]"
])

print("Done — now re-run the baseline cell, no restart needed this time.")

## 3) Ortam (.env) Özelleştirmeleri

`.env` dosyası repodan okunur. İstersen Colab üzerinden hızlıca override edebilirsin.

In [ ]:
import os

os.environ["DEVICE"] = "cuda"  # GPU yoksa CPU moduna düşer
os.environ["RAY_NUM_GPUS"] = "0.2"

# .env dosyasını ezmek istersen buraları aç:
# os.environ["NUM_CLIENTS"] = "10"
# os.environ["NUM_ROUNDS"] = "5"

print("DEVICE overwrite:", os.environ.get("DEVICE"))

## 4) RUN PFL BASELINE
Sadece run_pfl.py'yi koşturur. Çıktılar `outputs/metrics/simulation_results.csv` içerisine yazılır.

In [ ]:
import os
import subprocess
import sys

root = os.environ.get("PROJECT_ROOT", "/content/privproject")
print("=== RUNNING BASELINE PFL ENGINE ===")

process = subprocess.Popen(
    [sys.executable, "run_pfl.py"],
    cwd=root,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

process.wait()
print(f"\nReturn code: {process.returncode}")

## 5) Sonuçları indir
`outputs/` klasörünü paketler ve .zip olarak Colab'dan bilgisayarınıza kaydeder.

In [ ]:
import os
import subprocess
from google.colab import files

root = os.environ.get("PROJECT_ROOT", "/content/privproject")
zip_path = "/content/baseline_outputs.zip"
if os.path.isfile(zip_path):
    os.remove(zip_path)

subprocess.check_call(
    ["zip", "-r", zip_path, "outputs", "-x", "*.pyc", "-x", "*__pycache__*"],
    cwd=root,
)
files.download(zip_path)